# **KKBOX Churn Prediction And Retention Intelligence System - Streamlit Integration (Sample Input)**

---

## **1. Objective**

This notebook evaluates and operationalizes the machine learning models used in the KKBOX Churn Prediction and Retention Intelligence System.

The goal is to:

- Load and validate the final trained churn model

- Define the expected feature schema for safe, consistent scoring

- Implement prediction, risk labeling, and business‑value calculations

- Build a recommendation layer that translates model outputs into retention actions

- Provide a complete scoring pipeline for both single‑customer and batch use cases

At this stage, the focus is on model inference, evaluation, and deployment readiness rather than training.
The outcome is a fully functional scoring engine that powers the Retention OS and supports real‑time and batch churn prediction.

----

## **2. Setup**

In [1]:
# 1. Core Python Libraries
import os
import pandas as pd
import numpy as np

# 2. Visualization
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# 3. Scikit-Learn: Model Evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# 4. Scikit-Learn: Preprocessing
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 5. Baseline Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB

# 6. Advanced Models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import xgboost

# 7. Model Persistence
import pickle

# 8. Display Settings
pd.set_option("display.max_columns", None)

---

# **3. Load Final Model**

In [2]:
# Load the final trained model (pipeline + preprocessor + HGB model)
current_dir = os.getcwd()

while not os.path.exists(os.path.join(current_dir, "data", "processed")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/processed not found")
    current_dir = parent

os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/processed exists:", os.path.exists("data/processed"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


with open("model/final_churn_model.pkl", "rb") as f:
    model = pickle.load(f)
    

Project root: c:\Users\pauli\OneDrive\Documentos\GitHub\customer-churn-intelligence-system
data/processed exists: True


-----

# **4. Define Expected Feature Schema**

In [3]:
expected_features = [
    "gender", "age", "city_grouped", "registered_via_grouped",
    "avg_amount_paid", "total_amount_paid",
    "has_auto_renew", "has_cancelled",
    "total_secs", "num_unq",
    "customer_tenure_days", "listening_group",
    "payment_variability"
]

expected_features

['gender',
 'age',
 'city_grouped',
 'registered_via_grouped',
 'avg_amount_paid',
 'total_amount_paid',
 'has_auto_renew',
 'has_cancelled',
 'total_secs',
 'num_unq',
 'customer_tenure_days',
 'listening_group',
 'payment_variability']

-----

# **5. Imput Validation**

In [4]:
def validate_input(df):
   
    for col in expected_features:
        if col not in df.columns:
            df[col] = np.nan
    return df[expected_features]

------

# **6. Prediction Function**

In [5]:
def predict_churn(input_data):
   
    if isinstance(input_data, dict):
        df = pd.DataFrame([input_data])
    else:
        df = input_data.copy()

    df = validate_input(df)

    # Predict churn probability
    prob = model.predict_proba(df)[0][1]

    # Risk label
    if prob < 0.10:
        risk = "Low"
    elif prob < 0.30:
        risk = "Medium"
    else:
        risk = "High"

    return prob, risk

----

# **7. Business Value Layer**

In [6]:
def compute_business_metrics(input_data, churn_prob):

    avg_amount = input_data.get("avg_amount_paid", 0)
    total_amount = input_data.get("total_amount_paid", 0)

    arpu = avg_amount  # ARPU proxy

    churn_rate = max(churn_prob, 0.01) # LTV approximation 
    ltv = arpu * (1 / churn_rate)

    revenue_at_risk = arpu if churn_prob > 0.5 else arpu * churn_prob # Revenue at risk

    potential_saved = revenue_at_risk * 0.10    # Potential revenue saved (10% retention uplift)

    return {
        "ARPU": arpu,
        "LTV": ltv,
        "Revenue_at_Risk": revenue_at_risk,
        "Potential_Revenue_Saved": potential_saved
    }

-----

# **8. Recommendation Engine**

In [7]:
def recommend_action(input_data, risk_label):

    cancelled = input_data.get("has_cancelled", 0) # Generates a retention recommendation based on behavior + risk.
    auto_renew = input_data.get("has_auto_renew", 0)
    listening = input_data.get("listening_group", "Medium")

    if cancelled == 1:
        return "Recovery / Win-back"
    if auto_renew == 0:
        return "Preventive Retention"
    if risk_label == "Low" and input_data.get("avg_amount_paid", 0) > 150:
        return "Protective Retention"
    if listening in ["Low", "Medium-Low"]:
        return "Re-engagement"

    return "Maintain / Upsell"

-----

# **9. Full Scoring**

In [8]:
def score_customer(input_dict):
    
    # Full scoring pipeline:
    # - churn probability
    # - risk label
    # - business metrics
    # - recommended action
    
    prob, risk = predict_churn(input_dict)
    business = compute_business_metrics(input_dict, prob)
    action = recommend_action(input_dict, risk)

    return {
        "churn_probability": prob,
        "risk_label": risk,
        "business_metrics": business,
        "recommended_action": action
    }

----

# **10. Sample Input**

In [9]:
sample_user = {
    "gender": "male",
    "age": 25,
    "city_grouped": 5,
    "registered_via_grouped": 3,
    "avg_amount_paid": 120,
    "total_amount_paid": 600,
    "has_auto_renew": 1,
    "has_cancelled": 0,
    "total_secs": 50000,
    "num_unq": 200,
    "customer_tenure_days": 800,
    "listening_group": "Medium-High",
    "payment_variability": 20
}

score_customer(sample_user)

c:\Users\pauli\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] Das System kann die angegebene Datei nicht finden
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\pauli\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\pauli\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\pauli\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^

{'churn_probability': np.float64(0.934134699139148),
 'risk_label': 'High',
 'business_metrics': {'ARPU': 120,
  'LTV': np.float64(128.46113104521865),
  'Revenue_at_Risk': 120,
  'Potential_Revenue_Saved': 12.0},
 'recommended_action': 'Maintain / Upsell'}

----

# **11. Streamlit App Prototype**

In [10]:
%%writefile app.py
import streamlit as st
import pandas as pd
import pickle

# Load model
with open("model/final_churn_model.pkl", "rb") as f:
    model = pickle.load(f)

st.title("KKBOX Churn Prediction & Retention Intelligence System")

st.subheader("Customer Input")

gender = st.selectbox("Gender", ["male", "female", "unknown"])
age = st.number_input("Age", 0, 100)
city = st.text_input("City Grouped")
via = st.text_input("Registered Via Grouped")
avg_paid = st.number_input("Avg Amount Paid", 0, 2000)
total_paid = st.number_input("Total Amount Paid", 0, 20000)
auto = st.selectbox("Has Auto-Renew", [0, 1])
cancel = st.selectbox("Has Cancelled", [0, 1])
secs = st.number_input("Total Listening Seconds", 0, 2000000)
unq = st.number_input("Unique Songs", 0, 20000)
tenure = st.number_input("Customer Tenure (days)", 0, 5000)
listen_group = st.selectbox("Listening Group", ["Low", "Medium-Low", "Medium-High", "High"])
variability = st.number_input("Payment Variability", 0, 500)

input_dict = {
    "gender": gender,
    "age": age,
    "city_grouped": city,
    "registered_via_grouped": via,
    "avg_amount_paid": avg_paid,
    "total_amount_paid": total_paid,
    "has_auto_renew": auto,
    "has_cancelled": cancel,
    "total_secs": secs,
    "num_unq": unq,
    "customer_tenure_days": tenure,
    "listening_group": listen_group,
    "payment_variability": variability
}

if st.button("Predict Churn"):
    df = pd.DataFrame([input_dict])
    prob = model.predict_proba(df)[0][1]
    st.write(f"### Churn Probability: {prob:.2%}")

Overwriting app.py


# **12. Prospective Churn & Revenue Forecast (Next 12 Months)**

In [15]:
# 12. Prospective Churn & Revenue Forecast (Next 12 Months)

# Load full customer dataset for forecasting
df_full = pd.read_csv("data/processed/df_clean.csv")

# Score all customers
df_full["churn_prob"] = model.predict_proba(df_full[expected_features])[:, 1]

# Current ARPU (euros per user)
ARPU = df_full["avg_amount_paid"].mean()

# Forecast churners next year (customers)
expected_churners = int(df_full["churn_prob"].mean() * len(df_full))

# Revenue at risk (euros)
revenue_at_risk = expected_churners * ARPU

# Scenario modeling (formatted with units)
scenarios = pd.DataFrame({
    "Scenario": ["Base", "Optimistic", "Pessimistic"],
    "Expected_Churners (customers)": [
        expected_churners,
        int(expected_churners * 0.9),
        int(expected_churners * 1.2)
    ],
    "Revenue_at_Risk (€)": [
        round(revenue_at_risk, 2),
        round(revenue_at_risk * 0.9, 2),
        round(revenue_at_risk * 1.2, 2)
    ],
    "Revenue_Saved (€)": [
        round(revenue_at_risk * 0.10, 2),
        round(revenue_at_risk * 0.15, 2),
        round(revenue_at_risk * 0.05, 2)
    ]
})

# Convert to integers where appropriate
scenarios["Revenue_at_Risk (€)"] = scenarios["Revenue_at_Risk (€)"].astype(int)
scenarios["Revenue_Saved (€)"] = scenarios["Revenue_Saved (€)"].astype(int)

scenarios

,Scenario,Expected_Churners (customers),Revenue_at_Risk (€),Revenue_Saved (€)
0,Base,87387,11433336,1143333
1,Optimistic,78648,10290002,1715000
2,Pessimistic,104864,13720003,571666


**Insight**

- Retention = instant ROI → saving even 10–15% brings back €1M–€1.7M.

- Takeaway → Churn isn’t a minor operational issue, it’s a multi‑million‑euro revenue leak that targeted retention can quickly contain.